<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 2 &middot; Week 4 &middot; Part 2 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Coffee Roasting in TensorFlow

Part 1 set every weight by hand. Here the network has to find its own.

The task is the lecture's coffee-roasting example. Beans roasted too cool or too briefly are
undercooked; roasted too hot for too long, they burn. The good roasts occupy a triangle in the
temperature&ndash;duration plane, and you will see that a single neuron &mdash; logistic
regression &mdash; cannot find it, while a network with three hidden units can, and does so by
inventing three features of its own.

Training uses two lines, `compile` and `fit`, that Week 5 takes apart. Today treat them as
given; the subject is what the trained network **computes**, not how it got there.

**Companion theory lecture:** CSE 815, Week 4, Part 2 (Neural Networks in Code), sections
"Inference in Code" and "Building a Neural Network".

## What you will be able to do

1. Show, with a number, why one logistic unit fails on data whose positive region is bounded by
   several lines.
2. Normalise inputs with a Keras `Normalization` layer, and check it against NumPy.
3. Build, compile and fit a $2 \to 3 \to 1$ network with `Sequential`.
4. Read a trained network's weights, and identify what each hidden unit has learned to detect.
5. Make predictions for new examples &mdash; including the normalisation step that is easy to
   forget.

---

## 1. Setup and the data

`load_coffee()` returns 200 roasting trials: temperature in &deg;C and duration in minutes, with
`y = 1` for a good roast. The rule that generated the labels is in the docstring; the network
will never see it.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow's start-up chatter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)
keras.utils.set_random_seed(816)      # seeds Python, NumPy and TensorFlow together

BLUE, TEAL, AMBER, ROSE, GRAY = "#123B63", "#1B7F79", "#C97B17", "#9B2C4B", "#4A5560"

print("NumPy", np.__version__, "  TensorFlow", tf.__version__)

def load_coffee(seed=816, m=200):
    """Roasting trials: temperature (deg C) and duration (min); y = 1 for a good roast.
    A roast is good inside the triangle with corners (175, 12), (175, 15) and (260, 12):
    hot enough, long enough, and not so hot for so long that it burns."""
    rng = np.random.default_rng(seed)
    temp = np.round(rng.uniform(150, 285, m), 1)
    dur = np.round(rng.uniform(11.5, 15.5, m), 2)
    good = (temp > 175) & (dur > 12) & (3 * temp + 85 * dur < 1800)
    return np.column_stack([temp, dur]), good.astype(float)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


X_cof, y_cof = load_coffee()
print("X_cof.shape:", X_cof.shape, f"  good roasts: {int(y_cof.sum())} of {len(y_cof)}")

In [ ]:
def plot_roasts(ax, X, y):
    good = y == 1
    ax.scatter(X[good, 0], X[good, 1], s=26, c=TEAL, marker="s", label="good roast")
    ax.scatter(X[~good, 0], X[~good, 1], s=26, c=ROSE, marker="x", label="bad roast")
    ax.set_xlabel("temperature (deg C)")
    ax.set_ylabel("duration (min)")


fig, ax = plt.subplots(figsize=(6.4, 4.2))
plot_roasts(ax, X_cof, y_cof)
ax.plot([175, 175, 260, 175], [12, 15, 12, 12], c=GRAY, ls="--", lw=1, label="true region")
ax.set_title("200 roasting trials")
ax.legend(fontsize=8)
plt.show()

base_rate = 1 - y_cof.mean()
print(f"good roasts: {int(y_cof.sum())} of {len(y_cof)}")
print(f"a 'model' that always says 'bad' is right {base_rate:.2%} of the time -- remember this.")

---

## 2. Normalise the inputs

Temperature runs from 150 to 285; duration from 11.5 to 15.5. As Week 2 showed, features on
such different scales make gradient descent crawl. Keras provides a `Normalization` layer that
learns each feature's mean and variance from the data (`adapt`) and then maps
$x \mapsto (x - \mu)/\sigma$.

In [ ]:
norm_l = keras.layers.Normalization(axis=-1)
norm_l.adapt(X_cof)                               # learn mean and variance from the data
Xn = norm_l(X_cof).numpy()

mu = norm_l.mean.numpy().ravel()
sigma = np.sqrt(norm_l.variance.numpy().ravel())
print("learned mean    :", mu)
print("learned std dev :", sigma)
print(f"\ntemperature: {X_cof[:, 0].min():.1f} .. {X_cof[:, 0].max():.1f}  ->  "
      f"{Xn[:, 0].min():.3f} .. {Xn[:, 0].max():.3f}")
print(f"duration   : {X_cof[:, 1].min():.2f} .. {X_cof[:, 1].max():.2f}  ->  "
      f"{Xn[:, 1].min():.3f} .. {Xn[:, 1].max():.3f}")

# It is exactly Week 2's z-score normalisation (population standard deviation, ddof = 0).
Xn_np = (X_cof - X_cof.mean(axis=0)) / X_cof.std(axis=0)
assert np.allclose(Xn, Xn_np, atol=1e-5)
print("\nThe Normalization layer agrees with NumPy's z-score.")

> **Save the scaler.** `mu` and `sigma` are now part of the model. Every future input must be
> normalised with *these* numbers, not with statistics of the new data. Section 6 shows what
> happens if you forget.

---

## 3. One neuron is not enough

Before building a network, try the model you already know. A single sigmoid unit draws one
straight boundary. The good region has three.

In [ ]:
from sklearn.linear_model import LogisticRegression

one_neuron = LogisticRegression(C=1e6, max_iter=10000).fit(Xn, y_cof)   # C large: no penalty
acc_one = one_neuron.score(Xn, y_cof)
print(f"logistic regression accuracy: {acc_one:.4f}")
print(f"always saying 'bad'         : {base_rate:.4f}")

assert acc_one < base_rate
print("\nThe best single line is worse than ignoring the inputs altogether. No straight")
print("boundary has the good triangle on one side and all the bad roasts on the other.")

---

## 4. Build the network

Three hidden units &mdash; one for each edge of the triangle, if all goes well &mdash; and one
output unit, all sigmoid. This is the lecture's code, with names on the layers so that you can
fetch them later.

In [ ]:
keras.utils.set_random_seed(816)      # reproducible initial weights

model = Sequential([
    Input(shape=(2,)),
    Dense(units=3, activation="sigmoid", name="layer1"),
    Dense(units=1, activation="sigmoid", name="layer2"),
])
model.summary()

assert model.count_params() == 2 * 3 + 3 + 3 * 1 + 1 == 13

W1_init, b1_init = model.get_layer("layer1").get_weights()
print("layer 1 weights before training (random):\n", W1_init)
print("layer 1 biases before training (zero):", b1_init)

---

## 5. Train it

The two lines below are Week 5's subject. In brief: `compile` names the cost function
&mdash; `BinaryCrossentropy` is exactly the logistic loss of Week 3 &mdash; and the optimiser,
`Adam`, a refinement of gradient descent. `fit` runs the optimiser.

200 examples is a small data set, so we copy it 1000 times (`np.tile`) to give the optimiser
more steps per epoch. That is a convenience for this lab, not standard practice.

In [ ]:
Xt = np.tile(Xn, (1000, 1))
yt = np.tile(y_cof, 1000)
print("training copies:", Xt.shape, yt.shape)

model.compile(
    loss=keras.losses.BinaryCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
)
history = model.fit(Xt, yt, epochs=10, verbose=0)

for epoch, loss in enumerate(history.history["loss"], start=1):
    print(f"epoch {epoch:2d}   cost {loss:.6f}")

In [ ]:
p_train = model.predict(Xn, verbose=0)[:, 0]
acc_net = ((p_train >= 0.5) == y_cof).mean()
print(f"network accuracy on the 200 roasts: {acc_net:.4f}")
print(f"logistic regression               : {acc_one:.4f}")

# The reference run reaches 1.0000. Training from random weights can occasionally stall in a
# poorer solution (checkpoint 2 explores this), so the check here is deliberately looser.
assert acc_net >= 0.9, "training stalled -- re-run the build and train cells"

---

## 6. Predict new roasts

Two new roasts: 200&nbsp;&deg;C for 13.9 minutes, and 200&nbsp;&deg;C for 17 minutes. The
model was trained on normalised inputs, so the new inputs must pass through **the same**
`norm_l` first.

In [ ]:
X_new = np.array([[200.0, 13.9],
                  [200.0, 17.0]])                # one row per roast

p_new = model.predict(norm_l(X_new), verbose=0)
yhat_new = (p_new >= 0.5).astype(int)
for xi, p, yh in zip(X_new, p_new[:, 0], yhat_new[:, 0]):
    print(f"t = {xi[0]:.0f}, d = {xi[1]:.1f}:  P(good) = {p:.4f}  ->  {'good' if yh else 'bad'}")

assert yhat_new[0, 0] == 1 and yhat_new[1, 0] == 0
print("\nThe first is inside the triangle; the second has roasted too long and burnt.")

In [ ]:
# The classic mistake: forget to normalise.
p_raw = model.predict(X_new, verbose=0)
print("without normalisation:", p_raw.ravel())
print("\nTo the network, 200 means '200 standard deviations above the mean temperature'.")
print("Every hidden unit saturates, and the answer is meaningless -- and no error is raised.")

---

## 7. What did the hidden units learn?

Fetch the trained weights. Then, rather than staring at the numbers, look at what each hidden
unit **does**: plot its activation over the whole input plane.

In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1 =\n", W1, "\nb1 =", b1)
print("W2 =", W2.ravel(), "  b2 =", b2)
print("\nShapes: W1", W1.shape, " b1", b1.shape, " W2", W2.shape, " b2", b2.shape)
assert W1.shape == (2, 3) and W2.shape == (3, 1)

In [ ]:
def show_units(model, title):
    t = np.linspace(150, 285, 200)
    d = np.linspace(11.5, 15.5, 200)
    T, D = np.meshgrid(t, d)
    grid_n = norm_l(np.column_stack([T.ravel(), D.ravel()])).numpy()
    A1 = model.get_layer("layer1")(grid_n).numpy()
    A2 = model.predict(grid_n, verbose=0)
    panels = [A1[:, j] for j in range(A1.shape[1])] + [A2[:, 0]]
    names = [f"hidden unit {j}" for j in range(A1.shape[1])] + ["output"]
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.0))
    for ax, P, name in zip(axes, panels, names):
        ax.contourf(T, D, P.reshape(T.shape), levels=np.linspace(0, 1, 11), cmap="Blues")
        ax.contour(T, D, P.reshape(T.shape), levels=[0.5], colors=[AMBER], linewidths=1.5)
        good = y_cof == 1
        ax.scatter(X_cof[good, 0], X_cof[good, 1], s=8, c=TEAL, marker="s")
        ax.scatter(X_cof[~good, 0], X_cof[~good, 1], s=8, c=ROSE, marker="x")
        ax.set_title(name, fontsize=9)
        ax.set_xticks([175, 260])
        ax.set_yticks([12, 15])
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()


show_units(model, "Dark = activation near 1. Amber line = where the unit crosses 0.5.")

Each hidden unit's amber line is straight &mdash; one unit, one linear boundary &mdash; and each
lies along a different edge of the triangle. Make that precise: sort the bad roasts by *why*
they are bad, and measure each unit's average activation on each kind.

In [ ]:
t, d = X_cof[:, 0], X_cof[:, 1]
kinds = {
    "too cool   (t <= 175)": t <= 175,
    "too short  (d <= 12)": d <= 12,
    "burnt (3t + 85d >= 1800)": 3 * t + 85 * d >= 1800,
    "good roast": y_cof == 1,
}
A1_train = model.get_layer("layer1")(Xn).numpy()

print(f"{'kind of roast':26} {'n':>4}   mean activation of units 0, 1, 2")
print("-" * 70)
for name, sel in kinds.items():
    print(f"{name:26} {sel.sum():4d}   {A1_train[sel].mean(axis=0)}")

print()
for name, sel in list(kinds.items())[:3]:
    j = int(np.argmax(A1_train[sel].mean(axis=0)))
    print(f"'{name.split('(')[0].strip()}' is detected mainly by hidden unit {j}")

In the reference run each hidden unit fires strongly (mean activation above 0.8) for **one kind
of failure** and stays near zero on good roasts. Nobody told the network that there are three
ways to ruin coffee; it found them because three lines were needed to fence off the triangle.
Do not over-read this. A different random start can reach the same accuracy with a
differently organised hidden layer &mdash; checkpoint 2 shows one where a unit fires on the
*good* roasts and the output weights it positively. What does not change is that each hidden
unit contributes one straight boundary, and the output layer combines them.

Now look at the output layer's weights, `W2`. In the reference run all three are large and
**negative**, with a positive bias: the output unit says "good" only when *no* hidden unit is
firing. That is the mirror image of the network you will design by hand in Part 3, whose hidden
units detect *good* sides and whose output is an AND. Both compute the same function:
$\text{NOT}(\text{cool} \lor \text{short} \lor \text{burnt}) = \text{warm} \land \text{long}
\land \text{unburnt}$. Training does not care which of the two it finds.

In [ ]:
print("output weights W2:", W2.ravel(), "  bias b2:", b2)
print("\nOutput when no hidden unit fires  (a1 = 0, 0, 0):",
      model.get_layer("layer2")(np.zeros((1, 3))).numpy().ravel())
print("Output when one unit fires fully (a1 = 1, 0, 0):",
      model.get_layer("layer2")(np.array([[1.0, 0.0, 0.0]])).numpy().ravel())

### Checkpoint 1

Section 3 used scikit-learn for the single neuron. Do it in TensorFlow instead: build
`Sequential([Input(shape=(2,)), Dense(units=1, activation="sigmoid")])`, seed with
`keras.utils.set_random_seed(816)`, compile and fit exactly as in section 5, and report its
accuracy on the 200 roasts. How many roasts does it predict to be good?

*Expected:* accuracy about `0.68` (within a percentage point or two of scikit-learn's `0.70`;
both are below the `0.75` of always saying "bad"), with only about `20` roasts predicted good.
Explain in one sentence why a model with a worse accuracy than a constant can still be the
*best* straight line by the cost it minimises.

In [ ]:
# Your code here

### Checkpoint 2

Training starts from random weights, and not every start ends in the same place. Rebuild and
retrain the $2 \to 3 \to 1$ network three times, calling `keras.utils.set_random_seed(seed)`
before building, for `seed` in `[1, 2, 3]`. For each, report the final cost and the accuracy.
Print the per-kind activation table of section 7 for each run. For any run that does not reach
`1.0`, call `show_units` as well and describe what went wrong.

*Expected (reference machine):* seeds 1 and 3 reach accuracy `1.0` with final cost below
`0.001`; seed 2 stalls at about `0.945` with cost about `0.156`, because two of its hidden units
learn the *same* feature &mdash; their rows in the per-kind activation table are nearly
identical &mdash; so the network fences off only two edges of the triangle. Seeds 1 and 3 both
succeed, but organise their hidden units differently from section 7's run. Exact figures can differ
slightly between platforms; a stalled run among a few seeds is the expected pattern. Week 5
returns to this.

In [ ]:
# Your code here

---

## 8. Recap

- When the positive class is bounded by several lines, a single sigmoid unit fails: on the roasts
  it scores `0.70`, worse than the `0.75` of always saying "bad".
- `keras.layers.Normalization` learns $\mu$ and $\sigma$ with `adapt`; it is Week 2's z-score.
  New inputs must be normalised with the *same* layer, or predictions are silently meaningless.
- `Sequential([Input, Dense, Dense])` builds the network; `compile` and `fit` train it (Week 5).
  The $2 \to 3 \to 1$ network classifies all 200 roasts correctly in the reference run.
- In the reference run each hidden unit specialises in one failure &mdash; too cool, too short,
  or burnt &mdash; and the output says "good" only when none of them fires. These are features
  the network invented; another random start may organise them differently.
- Training from random weights can stall. Check the result, and when in doubt, retrain from
  another seed.

### Exercises to hand in

1. Retrain with **two** hidden units instead of three. Report the accuracy and use `show_units`
   to explain which edge the network gives up on, and why.
2. Retrain with **six** hidden units. Does accuracy change? Do the extra units find new
   features, duplicate old ones, or do nothing? Support your answer with the per-kind
   activation table.
3. Remove the normalisation (train on raw `X_cof`, tiled) with everything else unchanged. Report
   the cost after each epoch and the final accuracy, and relate what you see to Week 2's
   feature-scaling lecture.
4. Take the trained `W1`, `b1`, `W2`, `b2` and compute the network's output for the two roasts
   of section 6 **by hand in NumPy**, including the normalisation. Your numbers must match
   `model.predict`. (Part 3 builds the general tool for this.)

### Next

**Part 3 &mdash; Forward propagation in NumPy:** the same network with the library removed, a
roaster designed entirely by hand, and the matrix multiplication that makes it fast.